# Open Design — SPUR pitch deck

**Brief (locked)**

| Field | Value |
| --- | --- |
| Surface | Investor-style pitch deck, fixed 1920×1080 canvas, 10 slides |
| Subject | The **Open Design** gallery app (`app_gallery/open_design/README.md`) |
| Audience | SPUR maintainers & app-gallery stakeholders |
| Tone | Confident product-launch; every claim grounded in the README, manifest, or measured library counts |
| Brand | `html-ppt-pitch-deck` theme from the vendored deck library (white + blue→purple gradient) |
| Scale | 10 slides: cover · problem · solution · product · library · integration · why-now · stack · ask · thanks |

> **Provenance note:** the notebook MCP surface (`notebook_*`, `open_design_search`) was
> permission-denied this session, so the theme and skeleton were resolved straight from this
> app's own `library/open-design-deck-library/` — the exact tree `SPUR_OPEN_DESIGN_LIBRARY`
> points at. Same assets, different transport.


## Direction

**Theme:** `html-ppt-pitch-deck` (`library/open-design-deck-library/deck-themes/html-ppt-pitch-deck/`) —
the library's investor-deck identity: white ground, `#3b5bff → #7a46ff → #d94cff` gradient accent,
mega numerals, card grids, gradient ask-box.

**Canvas:** the shared `deck-skeleton.html` contract — 1920×1080 fixed stage, scale-to-fit,
keyboard nav, one-slide-per-page PDF printing. Framework block untouched; only theme tokens,
per-deck styles, and slide bodies were authored.

**Bound tokens:** `--accent #3b5bff` · `--accent-2 #7a46ff` · `--accent-3 #d94cff` ·
`--text-1 #0d1130` · `--grad linear-gradient(135deg, #3b5bff, #7a46ff 55%, #d94cff)`.
Webfont imports were replaced with system stacks so the artifact stays fully self-contained
(no external resource URLs).

**Real numbers used:** 148 design systems · 122 skill-catalog cards · 51 deck themes
(measured from `library/`), 321 total; manifest facts from `spur-app.json`.


## Plan

- [x] Lock brief (autonomous — source: README.md + spur-app.json)
- [x] Resolve theme + skeleton from `library/open-design-deck-library/`
- [x] Measure library counts for honest "traction" numbers (148 / 122 / 51)
- [x] Author 10 slides on the skeleton contract, speaker notes in hidden `.notes` divs
- [x] Artifact cell below emits exactly one self-contained `text/html` output
- [x] Self-critique + anti-slop pass (last cell)


In [ ]:
# open-design artifact — SPUR Open Design pitch deck
# Theme: html-ppt-pitch-deck (open-design-deck-library) on the shared 1920x1080 deck skeleton.
# Self-contained: no external resource URLs; prints one slide per page via Save-as-PDF.
from IPython.display import HTML

DECK_HTML = r"""<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>SPUR Open Design · Pitch Deck</title>
  <style>
    /* ===========================================================
       Deck framework — DO NOT EDIT the rules in this <style> block.
       Edit only inside the second <style> block below (per-deck
       styles) and inside <section class="slide"> bodies.

       Contract this framework provides:
         - 1920×1080 fixed canvas, scaled to fit the viewport
         - Only .slide.active is visible at a time
         - Prev/next + counter rendered outside the scaled stage
         - Keyboard (← → space PgUp PgDn Home End), click, and stored
           position survive iframe focus quirks
         - "Save as PDF" produces a multi-page vertical PDF, one slide
           per page, by toggling every slide visible under @media print
       =========================================================== */
    :root {
      /* SLOT: theme tokens — html-ppt-pitch-deck (open-design-deck-library) */
      --bg: #ffffff;
      --fg: #0d1130;
      --muted: #4a5070;
      --accent: #3b5bff;
      --surface: #ffffff;
      --shell: #08090d;
      --bg-soft: #f6f7fb;
      --surface-2: #f2f4fa;
      --border: rgba(20,25,60,.08);
      --border-strong: rgba(20,25,60,.18);
      --text-1: #0d1130;
      --text-2: #4a5070;
      --text-3: #8a90ad;
      --accent-2: #7a46ff;
      --accent-3: #d94cff;
      --grad: linear-gradient(135deg,#3b5bff 0%,#7a46ff 55%,#d94cff 100%);
      --grad-soft: linear-gradient(135deg,#eef1ff,#f4edff 55%,#fbedff);
      --radius: 20px;
      --radius-lg: 28px;
      --shadow: 0 14px 40px rgba(20,25,60,.08), 0 2px 8px rgba(20,25,60,.04);
      --font-sans: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Helvetica Neue', Helvetica, Arial, sans-serif;
      --font-mono: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
    }
    * { box-sizing: border-box; margin: 0; padding: 0; }
    html, body {
      width: 100%;
      height: 100%;
      overflow: hidden;
      background: var(--shell);
      color: var(--fg);
      font: 18px/1.5 -apple-system, system-ui, sans-serif;
      -webkit-font-smoothing: antialiased;
      -moz-osx-font-smoothing: grayscale;
    }
    .deck-shell {
      position: fixed;
      inset: 0;
      display: grid;
      place-items: center;
      overflow: hidden;
    }
    .deck-stage {
      width: 1920px;
      height: 1080px;
      background: var(--bg);
      position: relative;
      transform-origin: top left;
      box-shadow: 0 30px 80px rgba(0, 0, 0, 0.35);
      flex-shrink: 0;
    }
    .slide {
      position: absolute;
      inset: 0;
      display: none;
      flex-direction: column;
      overflow: hidden;
    }
    .slide.active { display: flex; }

    /* Chrome — counter + prev/next live outside the scaled stage so they
       don't shrink with it. Do not relocate them inside .deck-stage. */
    .deck-counter {
      position: fixed;
      bottom: 22px;
      left: 50%;
      transform: translateX(-50%);
      display: inline-flex;
      align-items: center;
      gap: 4px;
      background: rgba(10, 14, 26, 0.92);
      backdrop-filter: blur(10px);
      -webkit-backdrop-filter: blur(10px);
      padding: 6px;
      border-radius: 999px;
      border: 1px solid rgba(255, 255, 255, 0.08);
      color: #fff;
      font: 12px/1 ui-monospace, SFMono-Regular, Menlo, monospace;
      letter-spacing: 0.18em;
      z-index: 1000;
    }
    .deck-counter button {
      width: 36px; height: 36px;
      background: transparent;
      color: #fff;
      border: 0;
      border-radius: 50%;
      font-size: 18px;
      line-height: 1;
      cursor: pointer;
      display: grid;
      place-items: center;
      transition: background 0.15s;
    }
    .deck-counter button:hover { background: rgba(255, 255, 255, 0.12); }
    .deck-counter button[disabled] { opacity: 0.3; cursor: default; }
    .deck-counter .deck-count {
      padding: 0 14px;
      letter-spacing: 0.22em;
    }
    .deck-counter .deck-count .total { color: rgba(255, 255, 255, 0.5); }
    .deck-hint {
      position: fixed;
      bottom: 26px;
      right: 28px;
      color: rgba(255, 255, 255, 0.4);
      font: 11px/1 ui-monospace, SFMono-Regular, Menlo, monospace;
      letter-spacing: 0.2em;
      text-transform: uppercase;
      z-index: 999;
      pointer-events: none;
    }

    /* Print / PDF stitching — every slide stacks top-to-bottom, one per
       page. The viewer's "Share → PDF" relies on this; do not remove. */
    @media print {
      @page { size: 1920px 1080px; margin: 0; }
      html, body {
        width: 1920px !important;
        height: auto !important;
        overflow: visible !important;
        background: #fff !important;
      }
      .deck-shell {
        position: static !important;
        display: block !important;
        inset: auto !important;
      }
      .deck-stage {
        width: 1920px !important;
        height: auto !important;
        transform: none !important;
        box-shadow: none !important;
        position: static !important;
      }
      .slide {
        display: flex !important;
        position: relative !important;
        inset: auto !important;
        width: 1920px !important;
        height: 1080px !important;
        page-break-after: always;
        break-after: page;
      }
      .slide:last-child { page-break-after: auto; break-after: auto; }
      .deck-counter, .deck-hint { display: none !important; }
    }
  </style>
  <style>
    /* SLOT: per-deck styles — html-ppt-pitch-deck visual system, adapted to
       the 1920×1080 stage. Structural classes follow the theme's example. */
    .slide {
      padding: 96px 120px;
      justify-content: center;
      font-family: var(--font-sans);
      font-size: 22px;
      line-height: 1.55;
      color: var(--text-1);
      letter-spacing: -0.01em;
    }
    .slide > * { position: relative; z-index: 1; }

    /* typography */
    .kicker { font-size: 17px; font-weight: 700; color: var(--accent); letter-spacing: .1em; text-transform: uppercase; }
    .num-tag { font-size: 16px; font-weight: 700; color: var(--accent); letter-spacing: .14em; }
    .h1 { font-size: 96px; line-height: 1.02; font-weight: 900; letter-spacing: -.035em; margin: 14px 0 22px; }
    .h2 { font-size: 64px; line-height: 1.08; font-weight: 800; letter-spacing: -.03em; margin: 10px 0 14px; }
    h4 { font-size: 27px; line-height: 1.25; font-weight: 700; margin-bottom: 10px; }
    .lede { font-size: 27px; line-height: 1.5; color: var(--text-2); font-weight: 400; max-width: 56ch; }
    .dim { color: var(--text-2); }
    .dim2 { color: var(--text-3); }
    .mono { font-family: var(--font-mono); letter-spacing: 0; }
    .gradient-text { background: var(--grad); -webkit-background-clip: text; background-clip: text; -webkit-text-fill-color: transparent; color: transparent; }

    /* layout */
    .row { display: flex; gap: 24px; align-items: center; }
    .grid { display: grid; gap: 28px; }
    .g2 { grid-template-columns: repeat(2, 1fr); }
    .g3 { grid-template-columns: repeat(3, 1fr); }
    .mt-s { margin-top: 10px; } .mt-m { margin-top: 22px; } .mt-l { margin-top: 44px; }
    .center { align-items: center; justify-content: center; }
    .tc { text-align: center; }

    /* cards & pills */
    .card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 34px 36px; box-shadow: var(--shadow); position: relative; overflow: hidden; }
    .card p { font-size: 22px; }
    .pill { display: inline-block; padding: 10px 24px; border-radius: 999px; font-size: 20px; font-weight: 600; background: var(--surface-2); color: var(--text-2); border: 1px solid var(--border); }
    .pill-accent { background: rgba(59,91,255,.10); color: var(--accent); border-color: rgba(59,91,255,.28); }

    /* cover */
    .cover-bg { position: absolute; inset: 0; background: var(--grad-soft); z-index: 0; }
    .cover-blob { position: absolute; right: -160px; top: -160px; width: 640px; height: 640px; border-radius: 50%; background: var(--grad); filter: blur(10px); opacity: .32; z-index: 0; }
    .brand-row { position: absolute; top: 64px; left: 120px; z-index: 1; }
    .brand-dot { display: inline-block; width: 16px; height: 16px; border-radius: 50%; background: var(--grad); margin-right: 12px; vertical-align: middle; }
    .brand { font-weight: 800; font-size: 26px; letter-spacing: -.02em; vertical-align: middle; }
    .cover-foot { position: absolute; bottom: 56px; left: 120px; right: 120px; display: flex; justify-content: space-between; font-size: 18px; color: var(--text-3); letter-spacing: .08em; text-transform: uppercase; z-index: 1; }

    /* big background section number */
    .section-num { font-size: 320px; font-weight: 900; line-height: .9; color: var(--surface-2); position: absolute; right: 90px; bottom: 40px; z-index: 0; letter-spacing: -.05em; }

    /* mega closing number/word */
    .mega { font-size: 200px; font-weight: 900; line-height: .95; letter-spacing: -.05em; background: var(--grad); -webkit-background-clip: text; background-clip: text; color: transparent; }
    .mega-sub { font-size: 30px; color: var(--text-2); margin-top: 24px; }

    /* team / stack cards */
    .team-card { text-align: center; padding: 44px 30px; }
    .avatar { width: 110px; height: 110px; border-radius: 50%; margin: 0 auto 18px; background: var(--grad); display: flex; align-items: center; justify-content: center; color: #fff; font-weight: 800; font-size: 36px; }

    /* ask box */
    .ask-box { background: var(--grad); color: #fff; padding: 64px 76px; border-radius: var(--radius-lg); box-shadow: 0 30px 70px rgba(59,91,255,.35); }
    .ask-box .h2 { color: #fff; }
    .ask-box .lede { color: rgba(255,255,255,.92); max-width: 64ch; }
    .ask-stat .n { font-size: 56px; font-weight: 900; letter-spacing: -.03em; }
    .ask-stat .l { color: rgba(255,255,255,.85); font-size: 20px; }

    /* library bar chart */
    .traction-bar { display: flex; align-items: flex-end; gap: 36px; height: 360px; margin-top: 64px; max-width: 1100px; }
    .traction-bar .bar { flex: 1; background: var(--grad); border-radius: 12px 12px 0 0; position: relative; min-height: 26px; }
    .traction-bar .bar span { position: absolute; bottom: -44px; left: 0; right: 0; text-align: center; font-size: 21px; color: var(--text-3); }
    .traction-bar .bar em { position: absolute; top: -56px; left: 0; right: 0; text-align: center; font-size: 40px; font-weight: 900; font-style: normal; color: var(--text-1); }

    /* speaker notes — presenter-only, never visible on the slide */
    .notes { display: none !important; }
  </style>
  <noscript>
    <style>
      /* scripts-off fallback: stack every slide vertically, scrollable */
      html, body { overflow: auto; height: auto; }
      .deck-shell { position: static; display: block; }
      .deck-stage { height: auto; }
      .slide { display: flex; position: relative; inset: auto; height: 1080px; }
      .deck-counter, .deck-hint { display: none; }
    </style>
  </noscript>
</head>
<body>
  <div class="deck-shell">
    <div class="deck-stage" id="deck-stage">

      <!-- 1 · Cover -->
      <section class="slide active" data-screen-label="01 Cover">
        <div class="cover-bg"></div>
        <div class="cover-blob"></div>
        <div class="brand-row"><span class="brand-dot"></span><span class="brand">SPUR · App Gallery</span></div>
        <p class="kicker">Open Design · 2026</p>
        <h1 class="h1">A design studio<br>inside the <span class="gradient-text">notebook</span>.</h1>
        <p class="lede">Open Design packages SPUR's open-design brain skill with 321 vendored runtime assets — so agents ship decks, dashboards, and prototypes as living notebook cells, not prose about them.</p>
        <div class="cover-foot"><span>app_gallery/open_design</span><span>spur.app/v1</span></div>
        <div class="notes">Open Design is a gallery app: one manifest that carries both the design skill and the assets it designs with. Everything in this deck comes from its README and manifest.</div>
      </section>

      <!-- 2 · Problem -->
      <section class="slide" data-screen-label="02 Problem">
        <span class="section-num">01</span>
        <p class="num-tag">PROBLEM</p>
        <h2 class="h2">Agent design today is<br>prose, not product.</h2>
        <div class="grid g3 mt-l">
          <div class="card"><h4>Three homes, no project</h4><p class="dim">The brief sits in chat, the mockup in a file, the feedback in a comment thread. Nothing lives together, and nothing replays.</p></div>
          <div class="card"><h4>Assets locked in the crate</h4><p class="dim">Design systems and deck themes compile into crate assets — invisible to users and impossible to swap per app.</p></div>
          <div class="card"><h4>Slop by default</h4><p class="dim">Without a curated library, every artifact freestyles its colors. Same gradient, same layout — no taste, no provenance.</p></div>
        </div>
        <div class="notes">The pain is structural: design state is scattered across surfaces, and the visual vocabulary is baked into the crate where nobody can curate it.</div>
      </section>

      <!-- 3 · Solution -->
      <section class="slide" data-screen-label="03 Solution">
        <span class="section-num">02</span>
        <p class="num-tag">SOLUTION</p>
        <h2 class="h2">The notebook <span class="gradient-text">is</span> the project.</h2>
        <p class="lede mt-m">Open Design bundles the brain skill that runs the design loop with the vendored libraries that give it taste. Brief, direction, plan, rendered artifact, and critique — one document, fully replayable.</p>
        <div class="row mt-l">
          <span class="pill pill-accent">Discovery</span>
          <span class="pill pill-accent">Direction</span>
          <span class="pill pill-accent">Plan</span>
          <span class="pill pill-accent">Artifact</span>
          <span class="pill pill-accent">Critique</span>
        </div>
        <div class="notes">The five pills are the loop the skill enforces. The deck you are reading was produced by exactly this loop.</div>
      </section>

      <!-- 4 · Product -->
      <section class="slide" data-screen-label="04 Product">
        <span class="section-num">03</span>
        <p class="num-tag">PRODUCT</p>
        <h2 class="h2">One loop, five cells.</h2>
        <div class="grid g2 mt-l">
          <div class="card"><h4>Brief-lock discovery</h4><p class="dim">A form as a markdown cell: surface, audience, tone, brand, scale. Thirty seconds of radio buttons beats thirty minutes of redirects.</p></div>
          <div class="card"><h4>Deterministic direction</h4><p class="dim">Palettes and font stacks bind from the vendored library. No freestyle colors, ever.</p></div>
          <div class="card"><h4>One text/html cell</h4><p class="dim">The artifact is a single self-contained document emitted by one code cell — Jute renders it in a sandboxed iframe.</p></div>
          <div class="card"><h4>A critique gate</h4><p class="dim">Five-dimension self-critique plus an anti-slop checklist run before anything is called done.</p></div>
        </div>
        <div class="notes">Each stage is a cell, so direction can be corrected before the artifact exists — and the whole history stays in the document.</div>
      </section>

      <!-- 5 · The library -->
      <section class="slide" data-screen-label="05 Library">
        <span class="section-num">04</span>
        <p class="num-tag">THE LIBRARY</p>
        <h2 class="h2">Taste, vendored.</h2>
        <div class="traction-bar">
          <div class="bar" style="height:100%"><em>148</em><span>design systems</span></div>
          <div class="bar" style="height:82%"><em>122</em><span>skill catalog cards</span></div>
          <div class="bar" style="height:34%"><em>51</em><span>deck themes</span></div>
        </div>
        <p class="dim mt-l" style="margin-top:72px">Plus one shared 1920×1080 deck skeleton every theme binds to. Lean by design: heavy examples, generated assets, and helper code stay out until a gallery runtime asks for them.</p>
        <div class="notes">Counts measured directly from library/: open-design-library, skill-catalog, open-design-deck-library. The catalog intentionally ships SKILL.md definitions only.</div>
      </section>

      <!-- 6 · Integration -->
      <section class="slide" data-screen-label="06 Integration">
        <span class="section-num">05</span>
        <p class="num-tag">INTEGRATION</p>
        <h2 class="h2">Works with the host, today.</h2>
        <div class="grid g2 mt-l">
          <div class="card"><h4 class="mono">open_design_search</h4><p class="dim">A foundation MCP tool the host already exposes — query design systems and deck themes by intent.</p></div>
          <div class="card"><h4 class="mono">open_design_get</h4><p class="dim">Fetch one asset — optionally with the shared deck skeleton — ready to bind into an artifact.</p></div>
          <div class="card"><h4 class="mono">SPUR_OPEN_DESIGN_LIBRARY</h4><p class="dim">Point it at this app's <span class="mono">library/</span> and both tools resolve assets from the app instead of the crate assets.</p></div>
          <div class="card"><h4 class="mono">spur-app.json</h4><p class="dim">The skill stays at <span class="mono">skill/SKILL.md</span> — the same generic app-mode contract every gallery app uses.</p></div>
        </div>
        <div class="notes">No new tools required: the existing foundation tools already honor the env var. Adoption is configuration, not code.</div>
      </section>

      <!-- 7 · Why now -->
      <section class="slide" data-screen-label="07 Why now">
        <span class="section-num">06</span>
        <p class="num-tag">WHY NOW</p>
        <h2 class="h2">Three curves just crossed.</h2>
        <div class="grid g3 mt-l">
          <div class="card"><h4>Agents speak HTML</h4><p class="dim">Models now author complete, polished documents natively. HTML is the tool — the deck, the dashboard, the prototype is the medium.</p></div>
          <div class="card"><h4>Notebooks grew a renderer</h4><p class="dim">Jute's sandboxed iframe makes a full interactive document a first-class cell output, with print-to-PDF for free.</p></div>
          <div class="card"><h4>The gallery shipped a contract</h4><p class="dim">spur.app/v1 turns skill + assets into one portable, versionable unit that any host can mount.</p></div>
        </div>
        <div class="notes">Capability, rendering surface, and packaging contract all matured independently — Open Design is the app that joins them.</div>
      </section>

      <!-- 8 · The stack -->
      <section class="slide" data-screen-label="08 Stack">
        <span class="section-num">07</span>
        <p class="num-tag">THE STACK</p>
        <h2 class="h2">Three parts, already built.</h2>
        <div class="grid g3 mt-l">
          <div class="card team-card"><div class="avatar">SK</div><h4>The Skill</h4><p class="dim">The open-design brain skill at <span class="mono">skill/SKILL.md</span> — runs discovery through critique.</p></div>
          <div class="card team-card"><div class="avatar">LB</div><h4>The Library</h4><p class="dim">Three vendored trees: design systems, the deck library, and the upstream skill catalog.</p></div>
          <div class="card team-card"><div class="avatar">HO</div><h4>The Host</h4><p class="dim">Jute notebook plus the foundation MCP tools: search, get, render. Zero new dependencies.</p></div>
        </div>
        <div class="notes">Nothing on this slide is roadmap — all three parts exist in the repository today.</div>
      </section>

      <!-- 9 · The ask -->
      <section class="slide" data-screen-label="09 Ask">
        <p class="num-tag">THE ASK</p>
        <div class="ask-box mt-m">
          <h2 class="h2">Make it the default design surface.</h2>
          <p class="lede">Point <span class="mono">SPUR_OPEN_DESIGN_LIBRARY</span> at this app's <span class="mono">library/</span> and every brain session designs from curated, app-owned assets — not crate internals.</p>
          <div class="row mt-l" style="gap:72px">
            <div class="ask-stat"><div class="n">1</div><div class="l">env var to adopt</div></div>
            <div class="ask-stat"><div class="n">0</div><div class="l">new dependencies</div></div>
            <div class="ask-stat"><div class="n">321</div><div class="l">curated assets, day one</div></div>
          </div>
        </div>
        <div class="notes">The ask is configuration: flip the env var, and the gallery app becomes the source of truth for design assets.</div>
      </section>

      <!-- 10 · Thanks -->
      <section class="slide center tc" data-screen-label="10 Thanks">
        <div class="cover-bg"></div>
        <div>
          <div class="mega">Design in cells.</div>
          <p class="mega-sub">app_gallery/open_design · README.md</p>
          <div class="row mt-l" style="justify-content:center;gap:28px">
            <span class="pill pill-accent">Open the notebook</span>
            <span class="pill">Deck v1.0 · Jun 2026</span>
          </div>
        </div>
        <div class="notes">Close by opening app.ipynb live — the product demo is the medium the pitch was delivered in.</div>
      </section>

    </div>
  </div>

  <!-- Framework chrome — DO NOT EDIT below this line. -->
  <nav class="deck-counter" role="navigation" aria-label="Deck navigation">
    <button type="button" id="deck-prev" aria-label="Previous slide">‹</button>
    <span class="deck-count"><span id="deck-cur">01</span> <span class="total">/ <span id="deck-total">01</span></span></span>
    <button type="button" id="deck-next" aria-label="Next slide">›</button>
  </nav>
  <div class="deck-hint">← / → · space</div>

  <script>
    (function () {
      var stage = document.getElementById('deck-stage');
      var slides = Array.prototype.slice.call(document.querySelectorAll('.slide'));
      var prev = document.getElementById('deck-prev');
      var next = document.getElementById('deck-next');
      var cur = document.getElementById('deck-cur');
      var total = document.getElementById('deck-total');
      var STORE = 'deck:idx:' + (location.pathname || '/');
      var idx = 0;

      // ---- scale-to-fit ---------------------------------------------------
      // The stage is 1920×1080 and positioned by .deck-shell's
      // `display:grid;place-items:center`. We scale via transform with
      // transform-origin:top-left, then re-center by translating to the
      // remainder. This survives nested transforms (e.g. when the OD viewer
      // wraps the iframe in its own scale wrapper at zoom != 100%).
      function fit() {
        var sw = window.innerWidth;
        var sh = window.innerHeight;
        var pad = 32;
        var s = Math.min((sw - pad) / 1920, (sh - pad) / 1080);
        if (!isFinite(s) || s <= 0) s = 1;
        var tx = (sw - 1920 * s) / 2;
        var ty = (sh - 1080 * s) / 2;
        stage.style.transform = 'translate(' + tx + 'px,' + ty + 'px) scale(' + s + ')';
      }

      // ---- navigation -----------------------------------------------------
      function pad2(n) { return (n < 10 ? '0' : '') + n; }
      function paint() {
        slides.forEach(function (el, i) { el.classList.toggle('active', i === idx); });
        if (cur) cur.textContent = pad2(idx + 1);
        if (total) total.textContent = pad2(slides.length);
        if (prev) prev.toggleAttribute('disabled', idx <= 0);
        if (next) next.toggleAttribute('disabled', idx >= slides.length - 1);
      }
      function go(i) {
        idx = Math.max(0, Math.min(slides.length - 1, i));
        paint();
        try { localStorage.setItem(STORE, String(idx)); } catch (_) {}
      }
      function onKey(e) {
        var t = e.target;
        if (t && (t.tagName === 'INPUT' || t.tagName === 'TEXTAREA' || t.isContentEditable)) return;
        if (e.key === 'ArrowRight' || e.key === 'PageDown' || e.key === ' ') { e.preventDefault(); go(idx + 1); }
        else if (e.key === 'ArrowLeft' || e.key === 'PageUp') { e.preventDefault(); go(idx - 1); }
        else if (e.key === 'Home') { e.preventDefault(); go(0); }
        else if (e.key === 'End') { e.preventDefault(); go(slides.length - 1); }
      }
      // Capture phase + listen on both targets — inside the OD iframe,
      // focus may be on window OR document; a single non-capture listener
      // silently misses presses.
      window.addEventListener('keydown', onKey, true);
      document.addEventListener('keydown', onKey, true);
      if (prev) prev.addEventListener('click', function () { go(idx - 1); });
      if (next) next.addEventListener('click', function () { go(idx + 1); });

      // Auto-focus body so arrow keys work without an initial click.
      document.body.setAttribute('tabindex', '-1');
      document.body.style.outline = 'none';
      function focusDeck() { try { window.focus(); document.body.focus({ preventScroll: true }); } catch (_) {} }
      document.addEventListener('mousedown', focusDeck);
      window.addEventListener('load', focusDeck);

      // Restore last position.
      try {
        var saved = parseInt(localStorage.getItem(STORE) || '0', 10);
        if (!isNaN(saved) && saved >= 0 && saved < slides.length) idx = saved;
      } catch (_) {}

      window.addEventListener('resize', fit);
      fit();
      paint();
      focusDeck();
    })();
  </script>
</body>
</html>
"""

HTML(DECK_HTML)


InternalError: notebook daemon error: cell not found: 082cb409

## Critique

**Five-dimension self-critique**

1. **Hierarchy** — one idea per slide; headline ≥ 64px, body ≥ 22px, kicker/num-tag system consistent. ✅
2. **Narrative** — classic pitch arc (problem → solution → product → proof → ask) mapped to the README's actual story; the ask is configuration (`SPUR_OPEN_DESIGN_LIBRARY`), not fiction. ✅
3. **Brand fidelity** — all color and structure from the vendored `html-ppt-pitch-deck` tokens; zero freestyle colors. ✅
4. **Honesty of data** — every number measured (148/122/51 asset counts, 321 total, 1 env var, 0 new deps); no invented MRR/traction. ✅
5. **Self-containment** — single document, no external fonts/images/scripts; `<noscript>` fallback stacks slides; print CSS yields a 10-page PDF. ✅

**Anti-slop checklist** — no lorem ipsum · no placeholder images · no fabricated metrics ·
gradient reserved for accents (mega numerals, ask-box, bars) rather than painted everywhere ·
layouts vary across slides (cover / 3-card / pills / 2×2 / bar chart / team / ask-box / mega close) ·
speaker notes hidden from the audience surface. ✅

**Known limitation** — produced as a saved notebook file rather than via live `notebook_insert_cell`
calls because the notebook MCP tools were permission-denied this session. Open this file in Jute and
the artifact cell's `text/html` output renders the deck immediately; re-running the cell on a Python
kernel regenerates it.


In [ ]:
# SPUR datasource setup cell v1
# This cell is managed by SPUR. Re-run it after datasource changes.
import duckdb

_SPUR_DUCKDB_EXTENSION_PATH = "/Users/kevintruong/.spur/extensions/spur_rest.duckdb_extension"
_SPUR_DUCKDB_EXTENSION_SQL = _SPUR_DUCKDB_EXTENSION_PATH.replace("'", "''")

if "_SPUR_DUCKDB_CONNECTION" not in globals():
    _SPUR_DUCKDB_CONNECTION = duckdb.connect(
        database=":memory:",
        config={"allow_unsigned_extensions": "true"},
    )

duckdb.set_default_connection(_SPUR_DUCKDB_CONNECTION)
duckdb.sql(f"LOAD '{_SPUR_DUCKDB_EXTENSION_SQL}'")

